#### Time to run SFT on a larger 8B parameter model

In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
VRAM: 17.1 GB


In [1]:
!nvidia-smi

Fri Mar  6 16:51:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.57                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5080 ...    On  |   00000000:C2:00.0  On |                  N/A |
| N/A   38C    P4             26W /  175W |    1239MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Our System Prompt and also examples need to be refactored.

In [3]:
SYSTEM_PROMPT = """You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.

CONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.
- Answering without searching = WRONG
- Explaining instead of searching = WRONG
- Any text before your code block = WRONG

TOOLS:
- `context` - the document (already loaded, DO NOT redefine)
- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk

WORKFLOW:
1. Write ```python with print() to search
2. STOP immediately after code block
3. Wait for output (appears in next message)
4. Search more OR give FINAL(answer)

SEARCH STRATEGY:
- If find() returns -1, try DIFFERENT keywords (not the same one)
- Try simpler terms, synonyms, related concepts
- Read surrounding text with context[idx:idx+500] to understand what you found

DO NOT:
- Explain what you're doing
- Answer from memory
- Write multiple code blocks
- Add text before ```python
- Redefine the context variable

When done searching, end with: FINAL(your evidence-based answer)"""

#### We now need new examples that were trained with this.

In [5]:
!python /workspace/generate_training_data.py

Loaded 155 total traces
RLM Training Data Generator v2

--- Phase 1: Loading documents ---
Loading Gutenberg texts...
  [cavalry_outpost] cached: cavalry_outpost.txt
  [household_physician] cached: household_physician.txt
  [boston_cooking_school] cached: boston_cooking_school.txt
  [rules_of_road_sea] cached: rules_of_road_sea.txt
  [cushings_manual] cached: cushings_manual.txt
Loading 8 synthetic document(s)...
  [product_spec] written (3,243 chars)
  [lease_agreement] written (4,533 chars)
  [lab_report] written (3,980 chars)
  [meeting_minutes] written (5,053 chars)
  [employee_handbook] written (5,766 chars)
  [safety_data_sheet] written (5,276 chars)
  [grant_proposal] written (5,866 chars)
  [it_incident_report] written (6,528 chars)
Loading GPO style manual...
  [gpo_manual] loaded (147,114 chars)
Total documents loaded: 14

Loaded 14 documents:
  cavalry_outpost: 64,392 chars
  household_physician: 181,100 chars
  boston_cooking_school: 129,641 chars
  rules_of_road_sea: 25,01

In [6]:
import json

with open("/workspace/data/rlm_training_v2.json") as f:
    dataset = json.load(f)

print(f"Total examples: {len(dataset)}")
print(f"First example has {len(dataset[0]['messages'])} messages")

for msg in dataset[0]["messages"]:
    role = msg["role"]
    content = msg["content"][:120] + "..." if len(msg["content"]) > 120 else msg["content"] 
    print(f"\n[{role}]: {content}")

Total examples: 155
First example has 4 messages

[user]: You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must S...

[assistant]: I'll search for the keyword 'patrol' to find relevant passages.

```python
idx = context.find('patrol')
print(f'Found at...

[user]: Output:
Found at position: 7111
patrols to; whether he thinks that the

 Picquet has been well posted, at night as well ...

[assistant]: FINAL(The manual instructs that patrols should be sent out at irregular intervals along routes that cover the front and ...


In [9]:
from transformers import AutoTokenizer
import numpy as np
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.1-8B-Instruct") #loads the tokenizer
lengths = []
for example in dataset:
    lengths.append(len(tokenizer.apply_chat_template(example["messages"], tokenize = True)))
data = np.array(lengths)
np.max(data)

np.int64(843)

In [10]:
import unsloth
from unsloth import FastLanguageModel
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Llama-3.1-8B-Instruct", max_seq_length, load_in_4bit = True)

/tmp/ipykernel_20/3499575541.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 03-06 17:18:41 [interface.py:409] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [12]:
model = FastLanguageModel.get_peft_model(model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"], lora_alpha = 16, lora_dropout = 0) 

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [13]:
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [14]:
from datasets import Dataset
training_dataset = Dataset.from_list(dataset)
print(training_dataset)

Dataset({
    features: ['messages'],
    num_rows: 155
})


In [15]:
from trl import SFTTrainer, SFTConfig

def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

training_dataset = training_dataset.map(format_example)
training_args = SFTConfig(
    per_device_train_batch_size = 4,
    num_train_epochs = 4,
    learning_rate = 2e-4,
    max_seq_length = 1024,
    output_dir = "/workspace/outputs",
    logging_steps = 5
)

trainer = SFTTrainer(
    model = model,
    train_dataset = training_dataset,
    tokenizer = tokenizer,
    args = training_args,
)

Map:   0%|          | 0/155 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/155 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [16]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 155 | Num Epochs = 4 | Total steps = 80
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.077600
10,2.629800
15,2.028700
20,1.468300
25,0.964800
30,0.744800
35,0.734400
40,0.676400
45,0.712600
50,0.582400


TrainOutput(global_step=80, training_loss=1.0812831103801728, metrics={'train_runtime': 313.3314, 'train_samples_per_second': 1.979, 'train_steps_per_second': 0.255, 'total_flos': 1.643207515570176e+16, 'train_loss': 1.0812831103801728, 'epoch': 4.0})

In [17]:
FastLanguageModel.for_inference(model);
messages = [
    {"role": "user", "content": "You are a SEARCH assistant with a Python REPL. You search documents - nothing else.\n\nOUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.\n\nCONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.\n- Answering without searching = WRONG\n- Explaining instead of searching = WRONG\n- Any text before your code block = WRONG\n\nTOOLS:\n- `context` - the document (already loaded, DO NOT redefine)\n- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk\n\nWORKFLOW:\n1. Write ```python with print() to search\n2. STOP immediately after code block\n3. Wait for output (appears in next message)\n4. Search more OR give FINAL(answer)\n\nWhen done searching, end with: FINAL(your evidence-based answer)\n\n---\n\nWhat is the maximum occupancy for the building?"}
] # example it hasn't seen yet

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
outputs = model.generate(inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


```python
occupancy_idx = context.find('occupancy')
if occupancy_idx == -1:
    occupancy_idx = context.find('Occupancy')
print(f'Occupancy at: {occupancy_idx}')
```


#### day and night difference compared to 1B, listed below.

```python
occupancy_idx = context.find('occupancy')
if occupancy_idx == -1:
    occupancy_idx = context.find('Occupancy')
print(f'Occupancy at: {occupancy_idx}')
```

VS
    
```python
idx = context.find('building')
if idx == -1:
    idx = context.find('building')
print(f'idx: {idx}')
print(context[idx:idx+200])
```

In [19]:
model.save_pretrained("/workspace/outputs/8b/rlm_lora_v3")
tokenizer.save_pretrained("/workspace/outputs/8b/rlm_lora_v3")

('/workspace/outputs/8b/rlm_lora_v3/tokenizer_config.json',
 '/workspace/outputs/8b/rlm_lora_v3/special_tokens_map.json',
 '/workspace/outputs/8b/rlm_lora_v3/chat_template.jinja',
 '/workspace/outputs/8b/rlm_lora_v3/tokenizer.json')